In [9]:
import ijson
import json
import gc
from decimal import Decimal
from pathlib import Path

from sqlite_utils import Database
from tqdm import tqdm

from tuutrag.data import D

In [2]:
%load_ext autoreload
%autoreload 2

In [4]:
DB_PATH: Path          = D.processed / "master.db"
ALL_CHUNKS_PATH: Path  = D.processed / "all_chunks.json"
BRANCHES_PATH: Path    = D.processed / "branches.json"
LEAFS_PATH: Path       = D.processed / "leafs.json"
LEAF_EMBED_PATH: Path  = D.processed / "leaf_embed.json"

BATCH_SIZE: int = 5_000          # rows buffered before flushing to SQLite

# Remove stale DB
if DB_PATH.exists():
    DB_PATH.unlink()

db = Database(DB_PATH)

In [5]:
def artifact_uuid_from_branch(branch_uuid: str) -> str:
    """'...d300.1' -> '...d300'"""
    return branch_uuid.rsplit(".", 1)[0]

def branch_uuid_from_leaf(leaf_uuid: str) -> str:
    """'...d300.1.1' -> '...d300.1'"""
    parts = leaf_uuid.split(".")
    return f"{parts[0]}.{parts[1]}"

def serialize_embedding(embedding: list) -> str:
    """Decimal / float list  →  compact JSON string."""
    return json.dumps([float(v) for v in embedding])

In [6]:
db.executescript("""
    CREATE TABLE IF NOT EXISTS artifacts (
        uuid    TEXT PRIMARY KEY,
        path    TEXT NOT NULL DEFAULT '',
        type    TEXT NOT NULL DEFAULT '',
        summary TEXT
    );

    CREATE TABLE IF NOT EXISTS branches (
        uuid          TEXT PRIMARY KEY,
        artifact_uuid TEXT NOT NULL REFERENCES artifacts(uuid),
        chunk         TEXT NOT NULL DEFAULT '',
        path          TEXT NOT NULL DEFAULT '',
        summary       TEXT
    );

    CREATE TABLE IF NOT EXISTS branch_embeddings (
        branch_uuid TEXT PRIMARY KEY REFERENCES branches(uuid),
        embedding   TEXT NOT NULL
    );

    CREATE TABLE IF NOT EXISTS leafs (
        uuid        TEXT PRIMARY KEY,
        branch_uuid TEXT NOT NULL REFERENCES branches(uuid),
        text        TEXT NOT NULL DEFAULT '',
        entities    TEXT
    );

    CREATE TABLE IF NOT EXISTS leaf_embeddings (
        leaf_uuid TEXT PRIMARY KEY REFERENCES leafs(uuid),
        embedding TEXT NOT NULL
    );

    CREATE INDEX IF NOT EXISTS idx_branches_artifact ON branches(artifact_uuid);
    CREATE INDEX IF NOT EXISTS idx_leafs_branch      ON leafs(branch_uuid);
    CREATE INDEX IF NOT EXISTS idx_artifacts_type     ON artifacts(type);
    CREATE INDEX IF NOT EXISTS idx_artifacts_path     ON artifacts(path);
""")

print("Schema + indexes created.")

Schema + indexes created.


In [8]:
def stream_json_array(path: Path):
    """
    Yield one dict at a time from a top-level JSON array file
    using ijson's streaming parser.  Peak memory ≈ size of ONE object.
    """
    with open(path, "rb") as f:
        for obj in ijson.items(f, "item"):
            yield obj

def flush_batch(table_name: str, batch: list[dict], *, mode: str = "ignore") -> int:
    """
    Insert a batch of rows.  Uses INSERT OR IGNORE so duplicate
    PKs (e.g. duplicate artifact UUIDs across chunks) are silently
    skipped — no in-memory sets needed.
    """
    if not batch:
        return 0
    if mode == "ignore":
        db[table_name].insert_all(batch, ignore=True)
    elif mode == "replace":
        db[table_name].insert_all(batch, replace=True)
    n = len(batch)
    batch.clear()
    return n

In [10]:
artifact_buf: list[dict] = []
branch_buf: list[dict]   = []
total_chunks = 0

for obj in stream_json_array(ALL_CHUNKS_PATH):
    b_uuid = obj["uuid"]
    a_uuid = artifact_uuid_from_branch(b_uuid)

    artifact_buf.append({
        "uuid": a_uuid,
        "path":  obj.get("path", ""),
        "type":  obj.get("type", ""),
        "summary": None,
    })

    branch_buf.append({
        "uuid":          b_uuid,
        "artifact_uuid": a_uuid,
        "chunk":         obj.get("chunk", ""),
        "path":          obj.get("path", ""),
        "summary":       None,
    })

    total_chunks += 1

    if len(artifact_buf) >= BATCH_SIZE:
        flush_batch("artifacts", artifact_buf)
        flush_batch("branches",  branch_buf)
        gc.collect()

flush_batch("artifacts", artifact_buf)
flush_batch("branches",  branch_buf)
gc.collect()

print(f"Pass 1 done  — streamed {total_chunks:,} chunks  →  "
      f"artifacts: {db['artifacts'].count:,}  |  branches: {db['branches'].count:,}")

Pass 1 done  — streamed 37,278 chunks  →  artifacts: 453  |  branches: 37,239


In [11]:
branch_buf: list[dict]  = []
embed_buf: list[dict]   = []
total_branches = 0

for obj in stream_json_array(BRANCHES_PATH):
    b_uuid = obj["uuid"]
    a_uuid = artifact_uuid_from_branch(b_uuid)

    # Ensure parent artifact exists (in case branches.json
    # has entries not in all_chunks)
    artifact_buf.append({
        "uuid": a_uuid,
        "path":  obj.get("path", ""),
        "type":  "",
        "summary": None,
    })

    branch_buf.append({
        "uuid":          b_uuid,
        "artifact_uuid": a_uuid,
        "chunk":         obj.get("chunk", ""),
        "path":          obj.get("path", ""),
        "summary":       None,
    })

    if "embedding" in obj and obj["embedding"]:
        embed_buf.append({
            "branch_uuid": b_uuid,
            "embedding":   serialize_embedding(obj["embedding"]),
        })

    total_branches += 1

    if len(branch_buf) >= BATCH_SIZE:
        flush_batch("artifacts",        artifact_buf)
        flush_batch("branches",         branch_buf, mode="replace")
        flush_batch("branch_embeddings", embed_buf)
        gc.collect()

flush_batch("artifacts",        artifact_buf)
flush_batch("branches",         branch_buf, mode="replace")
flush_batch("branch_embeddings", embed_buf)
gc.collect()

print(f"Pass 2 done  — streamed {total_branches:,} branches  →  "
      f"branches: {db['branches'].count:,}  |  branch_embeddings: {db['branch_embeddings'].count:,}")

Pass 2 done  — streamed 37,278 branches  →  branches: 37,239  |  branch_embeddings: 37,239


In [12]:
leaf_buf: list[dict] = []
total_leafs = 0

for obj in stream_json_array(LEAFS_PATH):
    l_uuid = obj["uuid"]
    b_uuid = branch_uuid_from_leaf(l_uuid)

    leaf_buf.append({
        "uuid":        l_uuid,
        "branch_uuid": b_uuid,
        "text":        obj.get("text", ""),
        "entities":    None,
    })

    total_leafs += 1

    if len(leaf_buf) >= BATCH_SIZE:
        flush_batch("leafs", leaf_buf)
        gc.collect()

flush_batch("leafs", leaf_buf)
gc.collect()

print(f"Pass 3 done  — streamed {total_leafs:,} leafs  →  "
      f"leafs: {db['leafs'].count:,}")


Pass 3 done  — streamed 322,120 leafs  →  leafs: 322,120


In [13]:
leaf_backfill_buf: list[dict] = []
embed_buf: list[dict]         = []
total_leaf_embeds = 0

for obj in stream_json_array(LEAF_EMBED_PATH):
    l_uuid = obj["uuid"]
    b_uuid = branch_uuid_from_leaf(l_uuid)

    # Backfill leaf row if it wasn't in leafs.json
    leaf_backfill_buf.append({
        "uuid":        l_uuid,
        "branch_uuid": b_uuid,
        "text":        obj.get("text", ""),
        "entities":    None,
    })

    if "embedding" in obj and obj["embedding"]:
        embed_buf.append({
            "leaf_uuid": l_uuid,
            "embedding": serialize_embedding(obj["embedding"]),
        })

    total_leaf_embeds += 1

    if len(embed_buf) >= BATCH_SIZE:
        flush_batch("leafs",           leaf_backfill_buf)
        flush_batch("leaf_embeddings", embed_buf)
        gc.collect()

flush_batch("leafs",           leaf_backfill_buf)
flush_batch("leaf_embeddings", embed_buf)
gc.collect()

print(f"Pass 4 done  — streamed {total_leaf_embeds:,} leaf embeds  →  "
      f"leafs: {db['leafs'].count:,}  |  leaf_embeddings: {db['leaf_embeddings'].count:,}")

Pass 4 done  — streamed 322,120 leaf embeds  →  leafs: 322,120  |  leaf_embeddings: 322,120


In [14]:
print(f"\n{'═'*55}")
print(f"  Database: {DB_PATH}")
print(f"  Size:     {DB_PATH.stat().st_size / (1024**2):,.1f} MB")
print(f"{'═'*55}")
for table in db.table_names():
    print(f"  {table:25s} → {db[table].count:>10,} rows")

print(f"\nSchema:\n{db.schema}")


═══════════════════════════════════════════════════════
  Database: /Users/pablobedolla/DlrowSreckah/tuutrag-open/data/processed/master.db
  Size:     14,993.1 MB
═══════════════════════════════════════════════════════
  artifacts                 →        453 rows
  branches                  →     37,239 rows
  branch_embeddings         →     37,239 rows
  leafs                     →    322,120 rows
  leaf_embeddings           →    322,120 rows

Schema:
CREATE TABLE artifacts (
        uuid    TEXT PRIMARY KEY,
        path    TEXT NOT NULL DEFAULT '',
        type    TEXT NOT NULL DEFAULT '',
        summary TEXT
    );
CREATE TABLE branches (
        uuid          TEXT PRIMARY KEY,
        artifact_uuid TEXT NOT NULL REFERENCES artifacts(uuid),
        chunk         TEXT NOT NULL DEFAULT '',
        path          TEXT NOT NULL DEFAULT '',
        summary       TEXT
    );
CREATE TABLE branch_embeddings (
        branch_uuid TEXT PRIMARY KEY REFERENCES branches(uuid),
        embeddi